# Design A: one way of finding the points (D17)

Trains one of the three ways compared in `docs/decisions.md` (D17), exports it
to ONNX and writes its metrics. The dev sets and the test set are not used
here: the ONNX files are evaluated on the author's machine with
`python -m evaluation.run_dev ... --method way1 --models models/design_a`.

**Before running**
- Settings > Accelerator: a GPU (T4 or P100).
- Settings > Internet: on (GitHub, Hugging Face, Poly Haven, Roboflow).
- Add-ons > Secrets: `ROBOFLOW_API_KEY`, attached to this notebook.

**Runs**
1. `SMOKE = True`, any `WAY`: a few minutes, checks every step.
2. `SMOKE = False`, once per way: `way1`, `way2`, `way3` (up to 3 hours of
   training each). Use *Save Version > Save & Run All* so it runs in the
   background.

**Download** from the output: `<way>/model.onnx` and `<way>/metrics.json`,
into `models/design_a/<way>/` of the repository.

In [ ]:
WAY = "way1"  # way1, way2 or way3
COMMIT = "main"  # a commit of github.com/turhanGoksu/gaugelens
SMOKE = True  # True: a few steps on a few images; False: the real run
SYNTHETIC_TRAIN = 20000
SYNTHETIC_VAL = 1000
MAX_MINUTES = 180

In [ ]:
# Code and data live in /tmp so that Kaggle saves only the results.
import os
import subprocess

REPO = "/tmp/gaugelens"
URL = "https://github.com/turhanGoksu/gaugelens.git"
OUT = f"/kaggle/working/{WAY}"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", URL, REPO], check=True)
os.chdir(REPO)
# Fetch every time: a session that stays open must still see new pushes.
subprocess.run(["git", "fetch", "-q", "origin"], check=True)
branch = (
    subprocess.run(
        ["git", "rev-parse", "-q", "--verify", f"origin/{COMMIT}"], capture_output=True
    ).returncode
    == 0
)
subprocess.run(
    ["git", "checkout", "-q", f"origin/{COMMIT}" if branch else COMMIT], check=True
)
!git log --oneline -1
# Kaggle's numpy, OpenCV and PyTorch stay as they are; add only what is missing.
!pip install -q --no-deps -e .
!pip install -q onnxruntime onnx
import transformers

if not hasattr(transformers, "DFineForObjectDetection"):
    !pip install -q -U transformers
import torch
import torchvision

print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["ROBOFLOW_API_KEY"] = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")

In [ ]:
n_train, n_val = (64, 32) if SMOKE else (SYNTHETIC_TRAIN, SYNTHETIC_VAL)
# Steps whose output already exists are skipped, so this cell can run again.
# CC0 background photos from Poly Haven, then synthetic dials with exact labels.
if not os.path.exists("data/backgrounds"):
    !python -m gaugelens.synth download-backgrounds data/backgrounds
if not os.path.exists("data/synthetic/train"):
    !python -m gaugelens.synth generate data/synthetic/train --split train --count {n_train} --backgrounds data/backgrounds --workers 4
# Validation: the generator's dev backgrounds and its own seed range (D17).
if not os.path.exists("data/synthetic/val"):
    !python -m gaugelens.synth generate data/synthetic/val --split dev --seed-start 100100000 --count {n_val} --backgrounds data/backgrounds --workers 4
# Real images (D16). training.data checks every kept file against the
# SHA-256 in evaluation/train_roboflow/manifest.csv.
for project, version in [
    ("project-kaqwa/keypoints-4lma6", 1),
    ("computer-vision-ygqpn/gauge-reader-idem4", 11),
]:
    if not os.path.exists(f"data/external/roboflow/{project}/v{version}"):
        !python -m evaluation.roboflow download {project} {version} > /dev/null
!python -m training.data prepare data/prepared --synthetic data/synthetic/train --synthetic-val data/synthetic/val --roboflow data/external/roboflow

In [ ]:
flags = "--smoke" if SMOKE else f"--max-minutes {MAX_MINUTES}"
!python -m training.train {WAY} --data data/prepared --out {OUT} {flags}

In [ ]:
import json

metrics = json.load(open(f"{OUT}/metrics.json"))
fit = metrics["fit"]
print(
    f"{WAY} on {metrics['device']}: {fit['steps']} steps, {fit['seconds'] / 60:.0f} min"
)
print(
    "stopped by",
    fit["stopped"],
    "| best validation loss",
    round(fit["best_val_loss"], 4),
)
print("ONNX:", metrics["onnx"])
for kind, s in metrics["val_points"].items():
    print(f"\n{kind}: {s['dials']} dials")
    print("  error as a share of the dial radius: median / p90, missing")
    for key in ("center", "tip", "min", "max"):
        q = s[key]
        if q["median"] is None:
            print(f"  {key:6s} all missing")
        else:
            print(f"  {key:6s} {q['median']:.3f} / {q['p90']:.3f}, {q['missing']}")
!ls -la {OUT}